In [2]:
# Import data
import numpy as np
import pandas as pd
import pandas as pd
import requests
from pathlib import Path
import zipfile
import uuid
import json

url = "https://s3.amazonaws.com/dl.ncsbe.gov/data/ncvoter1.zip"
zip_path = Path("ncvoter1.zip")

r = requests.get(url)
zip_path.write_bytes(r.content)

zipfile.is_zipfile(zip_path)
extract_dir = Path("ncvoter1")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
	z.extractall(extract_dir)

list(extract_dir.iterdir())

df = pd.read_csv(
	extract_dir / 'ncvoter1.txt',
	sep='\t',
	encoding='latin1',
	# low_memory=False,
)

In [3]:
df.head()

,county_id,county_desc,voter_reg_num,ncid,last_name,first_name,middle_name,name_suffix_lbl,status_cd,voter_status_desc,...,sanit_dist_abbrv,sanit_dist_desc,rescue_dist_abbrv,rescue_dist_desc,munic_dist_abbrv,munic_dist_desc,dist_1_abbrv,dist_1_desc,vtd_abbrv,vtd_desc
0,1,ALAMANCE,9005990,AA56273,AABEL,RUTH,EVELYN,NaN,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,ALAMANCE,9178574,AA201627,AARDEN,JONI,AUTUMN,NaN,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,ALAMANCE,9205561,AA216996,AARMSTRONG,TIMOTHY,DUANE,NaN,I,INACTIVE,...,NaN,NaN,NaN,NaN,NaN,NaN,17.0,PROSECUTORIAL DISTRICT 17,103,103
3,1,ALAMANCE,9048723,AA98377,AARON,CHRISTINA,CASTAGNA,NaN,A,ACTIVE,...,NaN,NaN,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,03S,03S
4,1,ALAMANCE,9019674,AA69747,AARON,CLAUDIA,HAYDEN,NaN,A,ACTIVE,...,NaN,NaN,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124


In [4]:
df["label"] = 0
df["source"] = 'real'

In [5]:
df.columns

Index(['county_id', 'county_desc', 'voter_reg_num', 'ncid', 'last_name',
       'first_name', 'middle_name', 'name_suffix_lbl', 'status_cd',
       'voter_status_desc', 'reason_cd', 'voter_status_reason_desc',
       'res_street_address', 'res_city_desc', 'state_cd', 'zip_code',
       'mail_addr1', 'mail_addr2', 'mail_addr3', 'mail_addr4', 'mail_city',
       'mail_state', 'mail_zipcode', 'full_phone_number', 'confidential_ind',
       'registr_dt', 'race_code', 'ethnic_code', 'party_cd', 'gender_code',
       'birth_year', 'age_at_year_end', 'birth_state', 'drivers_lic', 'ssn',
       'no_dl_ssn_chkbx', 'hava_id_req', 'precinct_abbrv', 'precinct_desc',
       'municipality_abbrv', 'municipality_desc', 'ward_abbrv', 'ward_desc',
       'cong_dist_abbrv', 'super_court_abbrv', 'judic_dist_abbrv',
       'nc_senate_abbrv', 'nc_house_abbrv', 'county_commiss_abbrv',
       'county_commiss_desc', 'township_abbrv', 'township_desc',
       'school_dist_abbrv', 'school_dist_desc', 'fire_dist_a

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

In [7]:
!pip install -q transformers accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 55.5 MB/s eta 0:00:00


In [8]:
# If the model is gated
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Importing model

In [10]:
model_name = "Qwen/Qwen1.5-1.8B-Chat" # "Qwen/Qwen1.5-0.5B-Chat" for smaller

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

In [20]:
def generate_response(system_prompt, user_payload, max_new_tokens=1024, temperature=0.7):
    """
    Tokenizing the input prompt and user input
    """
    user_prompt = json.dumps(user_payload, ensure_ascii=False)
    full_prompt = (
        f"<|im_start|>system\n{system_prompt}\n<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1].strip()
    return text

In [21]:
def call_llm_for_variants(record, mode="train", n_variants=1):
    """
    Call LLM to generate variants of a record, return a list of variants.
    `record` is a dict (only the fields you want the LLM to see/perturb).
    mode: "train", "test", or "fraud"
    n_variants: number of variants to generate.
    """

    if mode == "train":
        system_prompt = (
            "You generate realistic variations of fictional customer records in a "
            "database for TRAINING a duplicate detection model.\n"
            "The user will send a JSON object with keys 'base_record' and 'n_variants'. "
            "'base_record' is one record; 'n_variants' is an integer. "
            "You MUST return exactly n_variants JSON records in a list.\n\n"
            "Constraints:\n"
            "- Each output record MUST represent the SAME fictional person as base_record.\n"
            "- Make small but realistic changes to the name and/or street address fields.\n"
            "- You MAY change several of these fields at once, but keep the person and "
            "location clearly the same.\n"
            "- DO NOT change city, zip_code, or birth_year.\n"
            "Return ONLY a JSON list of exactly n_variants records, each with exactly "
            "the same keys as base_record."
        )

    elif mode == "test":
        system_prompt = (
            "You generate more challenging, realistic variations of fictional customer "
            "records in a database for TESTING a duplicate detection model.\n"
            "The user will send a JSON object with keys 'base_record' and 'n_variants'. "
            "'base_record' is one record; 'n_variants' is an integer. "
            "You MUST return exactly n_variants JSON records in a list.\n\n"
            "Constraints:\n"
            "- Each output record MUST represent the SAME fictional person as base_record.\n"
            "- Make realistic changes to the name and/or street address fields that are "
            "generally larger or more complex than the changes used for training.\n"
            "- You MAY change multiple fields at once and combine several types of changes, "
            "as long as the person and location are still clearly the same.\n"
            "- City and zip_code should almost always stay the same; only change them if "
            "the full address is still clearly consistent with the same location.\n"
            "- You MAY change birth_year by at most ±1 in rare cases, but never more.\n"
            "Return ONLY a JSON list of exactly n_variants records, each with exactly "
            "the same keys as base_record."
        )

    elif mode == "fraud":
        system_prompt = (
            "You generate synthetic, internally inconsistent customer records for "
            "TESTING a fraud-detection model.\n"
            "These records are entirely fictional and will only be used to evaluate "
            "how well the model can detect bad or suspicious data.\n"
            "The user will send a JSON object with keys 'base_record' and 'n_variants'. "
            "You MUST return exactly n_variants JSON records in a list.\n\n"
            "Constraints:\n"
            "- Each output record must be PLAUSIBLE on the surface but contain one or more "
            "inconsistencies or suspicious patterns compared to base_record.\n"
            "- Do NOT create or modify any real identification numbers; treat all ID-like "
            "fields as opaque strings that should be copied unchanged from base_record.\n"
            "- The goal is to create synthetic bad data that LOOKS realistic but is "
            "internally inconsistent, so a fraud-detection model can learn to spot it.\n"
            "Return ONLY a JSON list of exactly n_variants records, each with exactly "
            "the same keys as base_record."
        )

    else:
        raise ValueError("mode must be 'train', 'test', or 'fraud'")

    user_payload = {"base_record": record, "n_variants": n_variants}

    response_text = generate_response(
        system_prompt=system_prompt,
        user_payload=user_payload,
        max_new_tokens=1024,
        temperature=0.7,
    )

    try:
        variants = json.loads(response_text)
    except json.JSONDecodeError:
        try:
            start = response_text.index("[")
            end = response_text.rindex("]") + 1
            variants = json.loads(response_text[start:end])
        except Exception:
            return []

    clean_variants = []
    for v in variants:
        if not isinstance(v, dict):
            continue

        new_v = {}
        for key in record.keys():
            val = v.get(key, record[key])
            new_v[key] = val

        clean_variants.append(new_v)

    return clean_variants[:n_variants]


In [22]:
def perturb_dataframe_with_llm(df_base, mode="train", frac=0.01, random_state=42):

    """
    Perturb a dataframe with LLM-generated variants.
    df_base is the dataframe to perturb.
    frac is the fraction of rows to perturb.
    """

    rng = np.random.default_rng(random_state)

    # sample seed rows
    df_seed = df_base.sample(frac=frac, random_state=random_state).reset_index(drop=True).copy()

    synthetic_rows = []

    for i, row in df_seed.iterrows():
        # Only 1 variant for robustness and distinctness
        n_variants = 1

        base_record = {
            "first_name": row["first_name"],
            "middle_name": row["middle_name"] if pd.notna(row["middle_name"]) else "",
            "last_name": row["last_name"],
            "name_suffix_lbl": row["name_suffix_lbl"] if pd.notna(row["name_suffix_lbl"]) else "",
            "res_street_address": row["res_street_address"],
            "res_city_desc": row["res_city_desc"],
            "zip_code": str(row["zip_code"]).split(".")[0] if pd.notna(row["zip_code"]) else "",
            "birth_year": int(row["birth_year"]) if pd.notna(row["birth_year"]) else None,
        }

        variants = call_llm_for_variants(base_record, mode=mode, n_variants=n_variants)

        for v in variants:
            # start from original row so we preserve all other columns (county, party, etc.)
            new_row = row.to_dict()
            # overwrite identity/location fields with perturbed values
            new_row["first_name"] = v["first_name"]
            new_row["middle_name"] = v.get("middle_name", "")
            new_row["last_name"] = v["last_name"]
            new_row["name_suffix_lbl"] = v.get("name_suffix_lbl", "")
            new_row["res_street_address"] = v["res_street_address"]
            new_row["res_city_desc"] = v["res_city_desc"]
            new_row["zip_code"] = v["zip_code"]
            new_row["birth_year"] = v["birth_year"]

            # mark as synthetic; label=1 because it is a duplicate of the seed row
            new_row["label"] = 1
            new_row["source"] = f"synthetic_{mode}"

            synthetic_rows.append(new_row)

    df_syn = pd.DataFrame(synthetic_rows)

    # seed rows are negatives by construction (no duplicates among themselves yet)
    df_seed = df_seed.copy()
    if "label" not in df_seed.columns:
        df_seed["label"] = 0
    if "source" not in df_seed.columns:
        df_seed["source"] = "real"

    df_aug = pd.concat([df_seed, df_syn], ignore_index=True)
    return df_seed, df_syn, df_aug

In [23]:
df_sample = df[:10]

df_real, df_syn, _ = perturb_dataframe_with_llm(df_sample, frac=0.8, mode='fraud')

In [25]:
df_real

,county_id,county_desc,voter_reg_num,ncid,last_name,first_name,middle_name,name_suffix_lbl,status_cd,voter_status_desc,...,rescue_dist_abbrv,rescue_dist_desc,munic_dist_abbrv,munic_dist_desc,dist_1_abbrv,dist_1_desc,vtd_abbrv,vtd_desc,label,source
0,1,ALAMANCE,9144384,AA125250,AARON,RICHARD,BRIAN,NaN,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,0,real
1,1,ALAMANCE,9178574,AA201627,AARDEN,JONI,AUTUMN,NaN,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,real
2,1,ALAMANCE,9129589,AA170513,AARON,JAMES,MICHAEL,NaN,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,real
3,1,ALAMANCE,9005990,AA56273,AABEL,RUTH,EVELYN,NaN,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,real
4,1,ALAMANCE,9041748,AA91549,AARON,NATHAN,EDWARD,NaN,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,03S,03S,0,real
5,1,ALAMANCE,9205561,AA216996,AARMSTRONG,TIMOTHY,DUANE,NaN,I,INACTIVE,...,NaN,NaN,NaN,NaN,17.0,PROSECUTORIAL DISTRICT 17,103,103,0,real
6,1,ALAMANCE,9144385,AA181361,AARON,SANDRA,ESCOBAR,NaN,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,0,real
7,1,ALAMANCE,9019674,AA69747,AARON,CLAUDIA,HAYDEN,NaN,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,0,real


In [26]:
df_syn

,county_id,county_desc,voter_reg_num,ncid,last_name,first_name,middle_name,name_suffix_lbl,status_cd,voter_status_desc,...,rescue_dist_abbrv,rescue_dist_desc,munic_dist_abbrv,munic_dist_desc,dist_1_abbrv,dist_1_desc,vtd_abbrv,vtd_desc,label,source
0,1,ALAMANCE,9144384,AA125250,AARON,RICHARD,BRIAN,,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,1,synthetic_fraud
1,1,ALAMANCE,9129589,AA170513,Aaron,James,Michael,,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,synthetic_fraud
2,1,ALAMANCE,9005990,AA56273,Aabel,Ruth,Evelyn,,R,REMOVED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,synthetic_fraud
3,1,ALAMANCE,9041748,AA91549,Aaron,Nathan,Edward,,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,03S,03S,1,synthetic_fraud
4,1,ALAMANCE,9205561,AA216996,AARMSTRONG,TIMOTHY,DUANE,,I,INACTIVE,...,NaN,NaN,NaN,NaN,17.0,PROSECUTORIAL DISTRICT 17,103,103,1,synthetic_fraud
5,1,ALAMANCE,9144385,AA181361,Aaron,Sandra,Escobar,,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,1,synthetic_fraud
6,1,ALAMANCE,9019674,AA69747,AARON,CLAUDIA,HAYDEN,,A,ACTIVE,...,NaN,NaN,BUR,BURLINGTON,17.0,PROSECUTORIAL DISTRICT 17,124,124,1,synthetic_fraud


In [27]:
ID_COLS = [
    "first_name", "middle_name", "last_name", "name_suffix_lbl",
    "res_street_address", "res_city_desc", "zip_code", "birth_year"
]

print("Identity columns for df_real (Original Records):")
display(df_real[ID_COLS].head())

print("\nIdentity columns for df_syn (Synthetic Fraud Records):")
display(df_syn[ID_COLS].head())

Identity columns for df_real (Original Records):


,first_name,middle_name,last_name,name_suffix_lbl,res_street_address,res_city_desc,zip_code,birth_year
0,RICHARD,BRIAN,AARON,NaN,1013 EDITH ST,BURLINGTON,27215.0,1972
1,JONI,AUTUMN,AARDEN,NaN,REMOVED,NaN,NaN,1978
2,JAMES,MICHAEL,AARON,NaN,REMOVED,NaN,NaN,1948
3,RUTH,EVELYN,AABEL,NaN,REMOVED,NaN,NaN,1935
4,NATHAN,EDWARD,AARON,NaN,421 WHITT AVE,BURLINGTON,27215.0,1976



Identity columns for df_syn (Synthetic Fraud Records):


,first_name,middle_name,last_name,name_suffix_lbl,res_street_address,res_city_desc,zip_code,birth_year
0,RICHARD,BRIAN,AARON,,1013 EDITH ST,BURLINGTON,27215,1972
1,James,Michael,Aaron,,REMOVED,NaN,,1948
2,Ruth,Evelyn,Aabel,,123 Main St,Anytown,12345,1935
3,Nathan,Edward,Aaron,,421 Whitt Ave.,Burlington,27215,1976
4,TIMOTHY,DUANE,AARMSTRONG,,3670 COVINGTON TRL,MEBANE,27302,1966


### To do:

1. Engineer better and diverse types of prompts. Seriously, what's 'anytown'?
2. Generate both positive and negative labels
3. Convert all the names to uppercase (Changing a name to lower case is not a valid perturbation)